In [1]:
import pandas as pd
df = pd.read_sas('/Users/tejaharshitamullapudi/Downloads/brfss-clinical-ai/data/raw/LLCP2024.XPT ', format='xport', encoding='utf-8')
print(df.shape)

(457670, 301)


In [2]:
KEEP_COLS = [
    '_STATE',
    'DIABETE4',
    'SMOKE100',
    'CVDINFR4',
    '_HLTHPL2',
    'MENTHLTH',
    'EXERANY2',
    'DRNKANY6',
    '_BMI5',
    'GENHLTH',
    '_SEX',
    '_AGEG5YR',
    '_RACE',
]

df_clean = df[KEEP_COLS].copy()
print(df_clean.shape)

(457670, 13)


In [3]:
df_clean['DIABETE4'].value_counts()


DIABETE4
3.0    376125
1.0     65809
4.0     11307
2.0      3395
7.0       798
9.0       232
Name: count, dtype: int64

In [4]:
df_clean = df_clean[~df_clean['DIABETE4'].isin([7.0, 9.0])]

In [5]:
df_clean['DIABETE4'].value_counts()

DIABETE4
3.0    376125
1.0     65809
4.0     11307
2.0      3395
Name: count, dtype: int64

In [6]:
invalid = [7.0, 9.0, 77.0, 99.0]

for col in KEEP_COLS:
    df_clean = df_clean[~df_clean[col].isin(invalid)]

print(df_clean.shape)

(299022, 13)


In [7]:
df_clean['diabetes'] = df_clean['DIABETE4'].map({
    1.0: 'Yes',
    2.0: 'Yes - pregnancy',
    3.0: 'No',
    4.0: 'Pre-diabetes'
})

df_clean['diabetes'].value_counts()

diabetes
No                 248158
Yes                 41585
Pre-diabetes         7006
Yes - pregnancy      2273
Name: count, dtype: int64

In [8]:
df_clean['smoking'] = df_clean['SMOKE100'].map({
    1.0: 'Yes',
    2.0: 'No'
})

df_clean['smoking'].value_counts()

smoking
No     183866
Yes    115156
Name: count, dtype: int64

In [9]:
df_clean['heart_attack'] = df_clean['CVDINFR4'].map({1.0: 'Yes', 2.0: 'No'})
df_clean['health_insurance'] = df_clean['_HLTHPL2'].map({1.0: 'Yes', 2.0: 'No'})
df_clean['exercise'] = df_clean['EXERANY2'].map({1.0: 'Yes', 2.0: 'No'})
df_clean['alcohol'] = df_clean['DRNKANY6'].map({1.0: 'Yes', 2.0: 'No'})
df_clean['general_health'] = df_clean['GENHLTH'].map({
    1.0: 'Excellent', 2.0: 'Very good', 3.0: 'Good', 4.0: 'Fair', 5.0: 'Poor'
})
df_clean['sex'] = df_clean['_SEX'].map({1.0: 'Male', 2.0: 'Female'})

print(df_clean.head())

   _STATE  DIABETE4  SMOKE100  CVDINFR4  _HLTHPL2  MENTHLTH  EXERANY2  \
0     1.0       3.0       2.0       2.0       1.0      88.0       1.0   
1     1.0       3.0       1.0       2.0       1.0      88.0       1.0   
2     1.0       3.0       1.0       2.0       1.0      88.0       1.0   
3     1.0       3.0       2.0       2.0       1.0      88.0       1.0   
4     1.0       3.0       2.0       2.0       1.0      88.0       2.0   

   DRNKANY6   _BMI5  GENHLTH  ...  _AGEG5YR  _RACE  diabetes smoking  \
0       2.0  2249.0      3.0  ...      12.0    1.0        No      No   
1       2.0  2583.0      1.0  ...      13.0    1.0        No     Yes   
2       1.0  2253.0      2.0  ...       8.0    1.0        No     Yes   
3       2.0  2509.0      1.0  ...      13.0    1.0        No      No   
4       2.0  1977.0      3.0  ...       6.0    1.0        No      No   

  heart_attack health_insurance exercise alcohol general_health     sex  
0           No              Yes      Yes      No      

In [10]:
import os
os.makedirs('data/processed', exist_ok=True)
df_clean.to_parquet('data/processed/brfss_clean.parquet')
print('saved!')


saved!


In [11]:
df = pd.read_parquet('data/processed/brfss_clean.parquet')

In [12]:
df.head()


,_STATE,DIABETE4,SMOKE100,CVDINFR4,_HLTHPL2,MENTHLTH,EXERANY2,DRNKANY6,_BMI5,GENHLTH,...,_AGEG5YR,_RACE,diabetes,smoking,heart_attack,health_insurance,exercise,alcohol,general_health,sex
0,1.0,3.0,2.0,2.0,1.0,88.0,1.0,2.0,2249.0,3.0,...,12.0,1.0,No,No,No,Yes,Yes,No,Good,Female
1,1.0,3.0,1.0,2.0,1.0,88.0,1.0,2.0,2583.0,1.0,...,13.0,1.0,No,Yes,No,Yes,Yes,No,Excellent,Male
2,1.0,3.0,1.0,2.0,1.0,88.0,1.0,1.0,2253.0,2.0,...,8.0,1.0,No,Yes,No,Yes,Yes,Yes,Very good,Male
3,1.0,3.0,2.0,2.0,1.0,88.0,1.0,2.0,2509.0,1.0,...,13.0,1.0,No,No,No,Yes,Yes,No,Excellent,Male
4,1.0,3.0,2.0,2.0,1.0,88.0,2.0,2.0,1977.0,3.0,...,6.0,1.0,No,No,No,Yes,No,No,Good,Male
